# 비용 함수 실습 · 한 셀씩 계산하며 이해하기

39페이지까지 배운 내용을 직접 확인합니다. **설명 읽기 → 짧은 코드 실행 → 출력 해석** 순서로 위에서 아래로 진행하세요. 처음에는 ‘모두 실행’보다 셀 왼쪽의 실행 버튼을 하나씩 누르는 것을 권합니다.

공개 자동차 데이터 8개로 실제값과 예측값을 비교합니다. 파랑 ●은 실제값, 주황 ×는 예측값, 보라 선은 둘 사이의 오차입니다.

마지막에는 Colab의 슬라이더와 재생 버튼으로 왼쪽 예측 함수와 오른쪽 비용 함수를 함께 봅니다. 별도 웹사이트나 영상은 사용하지 않습니다. CPU로 실행할 수 있습니다.

오늘의 질문은 **“이 직선이 얼마나 잘 맞는지를 어떻게 숫자 하나로 나타낼까?”**입니다.

## 0. 실습 준비
다음 셀은 그림과 한글 글꼴을 준비합니다. 처음 한 번만 실행하세요. 준비 코드는 접어 두었습니다. 수업의 핵심 계산은 아래에서 한 줄씩 진행합니다.

In [ ]:
# @title 실습 도구와 한글 글꼴 준비
import urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
import ipywidgets as widgets
from IPython.display import display
from google.colab import output

output.enable_custom_widget_manager()
font_path = Path('/content/NanumGothic-Regular.ttf')
if not font_path.exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', font_path)
font_manager.fontManager.addfont(str(font_path))
plt.rcParams['font.family'] = font_manager.FontProperties(fname=str(font_path)).get_name()
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110
BLUE, ORANGE, PURPLE, GREEN = '#1768ad', '#c86600', '#8c439c', '#207454'
print('준비 완료. 다음 셀로 이동하세요.')

## 1. 실제 데이터를 가져오기
자동차의 **무게로 연비를 예측**해 봅시다. 연비(mpg)가 클수록 같은 연료로 더 멀리 갑니다.

출처는 R. Quinlan (1993), [UCI Auto MPG](https://archive.ics.uci.edu/dataset/9/auto+mpg), [DOI 10.24432/C5859H](https://doi.org/10.24432/C5859H), CC BY 4.0입니다. 다운로드에는 [seaborn의 공개 사본](https://github.com/mwaskom/seaborn-data/blob/master/mpg.csv)을 사용합니다. 전체 398개를 무게순으로 정렬하고 고르게 떨어진 실제 관측치 8개를 선택합니다. 계산을 따라가기 위한 선택이며 성능 평가용 표본은 아닙니다.

In [ ]:
cars = pd.read_csv('https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv')
ordered = cars.sort_values('weight', kind='stable').reset_index(drop=True)
sample = ordered.iloc[np.rint(np.linspace(0, len(ordered) - 1, 8)).astype(int)][['name', 'weight', 'mpg']].copy()
sample.index = np.arange(1, 9)
display(sample.rename(columns={'name': '자동차', 'weight': '무게(lb)', 'mpg': '실제 연비(mpg)'}))

### 입력 $x$, 실제값 $y$, 데이터 수 $m$
$x$는 무게를 1,000파운드 단위로 바꾼 값입니다. 예를 들어 1,613파운드는 $x=1.613$입니다. $y$는 실제 연비, $m$은 자동차 수입니다. 아래 세 셀을 차례로 실행하세요.

In [ ]:
x = sample['weight'].to_numpy() / 1000
x

In [ ]:
y = sample['mpg'].to_numpy()
y

In [ ]:
m = len(y)
m

### 먼저 실제값만 보기
파란 점 하나가 자동차 한 대입니다. 가로축은 무게, 세로축은 실제 연비입니다. 무거운 차의 연비가 대체로 어떻게 되는지 말해 보세요.

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(x, y, color=BLUE, label='실제값', s=60)
plt.xlabel('무게 x (1,000파운드)')
plt.ylabel('연비 y (mpg)')
plt.legend()
plt.grid(alpha=.2)
plt.show()

## 2. 직선 하나로 예측하기
$$\hat y=f_{w,b}(x)=wx+b$$
기울기 $w$는 직선이 얼마나 기울어지는지, 절편 $b$는 위아래 위치를 정합니다. 일단 $w=-5$, $b=35$인 직선을 골라 봅시다. 아직 가장 좋은 직선이라는 뜻은 아닙니다.

In [ ]:
w, b = -5.0, 35.0

In [ ]:
y_pred = w * x + b
y_pred

예측값은 정답을 바꾸지 않습니다. 같은 자동차에 대해 “실제로는 얼마였나”와 “우리 직선은 얼마라고 예상했나”를 나란히 적습니다.

In [ ]:
comparison = pd.DataFrame({'무게 x': x, '실제 y': y, '예측값': y_pred}, index=sample.index)
display(comparison.round(3))

### 실제값과 예측값을 색으로 비교하기
같은 가로 위치의 파란 점과 주황 ×를 짝지어 보세요. 두 표시가 가까울수록 그 자동차의 예측은 정확합니다.

In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(x, y, color=BLUE, label='실제값', s=65)
plt.scatter(x, y_pred, color=ORANGE, marker='x', label='예측값', s=75)
plt.plot(x, y_pred, color=ORANGE, alpha=.7)
plt.xlabel('무게 x (1,000파운드)')
plt.ylabel('연비 (mpg)')
plt.legend()
plt.grid(alpha=.2)
plt.show()

## 3. 자동차마다 오차 구하기
$$e^{(i)}=\hat y^{(i)}-y^{(i)}$$
실제보다 크게 예측하면 양수, 작게 예측하면 음수입니다. 보라색 세로선의 **길이**는 오차의 크기이고, 오차의 부호는 예측이 위인지 아래인지를 알려줍니다.

In [ ]:
errors = y_pred - y
errors

In [ ]:
plt.figure(figsize=(8, 4))
plt.vlines(x, y, y_pred, color=PURPLE, linestyle='--', label='오차의 크기')
plt.scatter(x, y, color=BLUE, label='실제값', s=65)
plt.scatter(x, y_pred, color=ORANGE, marker='x', label='예측값', s=75)
plt.plot(x, y_pred, color=ORANGE, alpha=.6)
plt.xlabel('무게 x (1,000파운드)')
plt.ylabel('연비 (mpg)')
plt.legend()
plt.grid(alpha=.2)
plt.show()

### 오차를 그냥 더하면 왜 곤란할까요?
한 차를 5만큼 높게, 다른 차를 5만큼 낮게 예측하면 합은 0입니다. 그렇다고 두 차를 정확히 맞춘 것은 아닙니다. 시험에서 두 문제를 틀렸는데 서로 지워져 만점이 되는 것과 비슷합니다.

In [ ]:
np.array([5, -5]).sum()

## 4. 제곱해서 부호가 상쇄되지 않게 만들기
$$\left(\hat y^{(i)}-y^{(i)}\right)^2$$
$(-5)^2=25$이고 $5^2=25$입니다. 제곱은 큰 오차를 더 크게 반영합니다. 오차가 2배가 되면 제곱 오차는 4배가 됩니다.

In [ ]:
squared_errors = errors ** 2
squared_errors

In [ ]:
display(pd.DataFrame({'실제 y': y, '예측값': y_pred, '오차': errors, '제곱 오차': squared_errors}, index=sample.index).round(3))

## 5. $\sum$는 하나씩 더하라는 뜻
$$\sum_{i=1}^{m}\left(\hat y^{(i)}-y^{(i)}\right)^2$$
$i$는 자동차 번호입니다. 여기서는 1번부터 8번까지 제곱 오차를 모두 더합니다. 장바구니의 각 물건 가격을 더해 합계를 만드는 것처럼, 각 자동차의 제곱 오차를 모읍니다.

In [ ]:
running_total = np.cumsum(squared_errors)
display(pd.DataFrame({'이번 제곱 오차': squared_errors, '여기까지 합계': running_total}, index=sample.index).round(3))

### 한 대씩 더하는 과정을 직접 보기
아래 슬라이더의 **더한 차량 수**를 0부터 8까지 움직이세요. 재생 버튼으로 하나씩 볼 수도 있습니다. 연한 막대는 아직 더하지 않은 값입니다.

In [ ]:
# @title 제곱 오차를 하나씩 더해 보기
def show_sum(count):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    bars = ax.bar(np.arange(1, m + 1), squared_errors, color=GREEN)
    for i, bar in enumerate(bars):
        bar.set_alpha(1 if i < count else .15)
    ax.set(xlabel='자동차 번호 i', ylabel='제곱 오차', title=f'{count}대까지 더한 합계 = {squared_errors[:count].sum():.3f}')
    ax.set_xticks(np.arange(1, m + 1))
    plt.show()

count_slider = widgets.IntSlider(value=0, min=0, max=m, description='더한 차량 수', style={'description_width': 'initial'})
sum_play = widgets.Play(value=0, min=0, max=m, interval=1100)
sum_link = widgets.jslink((sum_play, 'value'), (count_slider, 'value'))
sum_output = widgets.interactive_output(show_sum, {'count': count_slider})
display(widgets.HBox([sum_play, count_slider]), sum_output)

In [ ]:
sum_squared_error = squared_errors.sum()
sum_squared_error

## 6. $m$으로 나누어 평균내기
$$\frac{1}{m}\sum_{i=1}^{m}\left(\hat y^{(i)}-y^{(i)}\right)^2$$
자동차 수가 많다는 이유만으로 점수가 커지지 않도록 자동차 한 대당 평균으로 바꿉니다. 똑같은 8대를 한 번 더 복사하면 합계와 자동차 수는 모두 두 배가 되지만 평균은 같습니다.

In [ ]:
mean_squared_error = sum_squared_error / m
mean_squared_error

## 7. 마지막으로 2로 나누면 비용 $J$
$$J(w,b)=\frac{1}{2m}\sum_{i=1}^{m}\left(f_{w,b}(x^{(i)})-y^{(i)}\right)^2$$
$\frac{1}{2m}$은 **$m$으로 나누고, 다시 2로 나누기**입니다. 여기서는 $m=8$이므로 분모가 16입니다. $1/2$는 나중에 미분식을 간단하게 만드는 관례입니다. 모든 점수를 똑같이 절반으로 줄이므로 어떤 직선이 가장 좋은지는 바뀌지 않습니다.

In [ ]:
J = mean_squared_error / 2
J

In [ ]:
display(pd.DataFrame({'계산 단계': ['제곱 오차의 합계', '8로 나눈 평균', '다시 2로 나눈 비용 J'], '값': [sum_squared_error, mean_squared_error, J]}))

## 8. 예측 함수와 비용 함수는 무엇이 다를까요?
$f_{w,b}(x)$는 **자동차 한 대의 예측 연비**입니다. $J(w,b)$는 **선택한 직선으로 모든 자동차를 예측한 결과를 평가하는 점수 하나**입니다.

아래는 지금까지 계산한 것을 함수 하나로 묶은 것입니다. 핵심 계산은 한 줄입니다.

In [ ]:
def cost(w, b):
    return np.mean((w * x + b - y) ** 2) / 2

cost(-5, 35)

## 9. $\min_w J(w)$ · 기울기만 고르기
먼저 $b=0$으로 고정합니다. 이때 $f_w(x)=wx$이고 모든 직선이 원점을 지납니다.

**왼쪽 가로축은 자동차 무게 $x$, 오른쪽 가로축은 기울기 $w$입니다.** 왼쪽 직선 하나가 오른쪽 비용 곡선 위 점 하나에 대응합니다. 오른쪽에서 점 하나의 높이를 계산할 때도 자동차 8대를 전부 사용합니다.

다음 셀은 그림을 그리는 준비 셀입니다. 그래프 작성 코드는 접혀 있고, 수업에서는 그 아래 슬라이더 셀을 사용하면 됩니다.

In [ ]:
# @title 좌우 그래프를 그리는 함수 준비
def draw_predictions(ax, slope, intercept):
    predicted = slope * x + intercept
    grid = np.linspace(0, 5.5, 100)
    ax.plot(grid, slope * grid + intercept, color=ORANGE)
    ax.vlines(x, y, predicted, color=PURPLE, linestyles='--', alpha=.65)
    ax.scatter(x, y, color=BLUE, label='실제값', s=45)
    ax.scatter(x, predicted, color=ORANGE, marker='x', label='예측값', s=50)
    ax.set(xlabel='입력 x · 무게', ylabel='연비', title=f'예측 함수: {slope:.2f}x + {intercept:.2f}')
    ax.legend()
    ax.grid(alpha=.2)

def compare_fixed(slope, intercept):
    candidates = np.linspace(-20, 20, 201)
    costs = np.array([cost(value, intercept) for value in candidates])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    draw_predictions(axes[0], slope, intercept)
    axes[1].plot(candidates, costs, color=GREEN)
    axes[1].scatter([slope], [cost(slope, intercept)], color=ORANGE, s=80, zorder=3)
    axes[1].set(xlabel='기울기 w', ylabel='전체 8대의 비용 J', title=f'b={intercept:.0f} 고정, 현재 J={cost(slope, intercept):.2f}')
    axes[1].grid(alpha=.2)
    plt.show()

### 실험 A · 기울기만 움직이기
기울기를 바꾸면서 왼쪽 직선과 오른쪽 주황점을 함께 보세요. 재생은 여러 기울기를 차례로 시험하는 것이며 경사 하강법은 아닙니다. 절편 선택을 45로 바꾸면 다른 비용 곡선이 나타납니다. **각 곡선 안에서는 절편이 고정**되어 있습니다.

In [ ]:
slope_slider = widgets.FloatSlider(value=6, min=-20, max=20, step=.5, description='기울기 w', continuous_update=True)
fixed_b = widgets.Dropdown(options=[('0으로 고정', 0), ('45로 고정', 45)], description='절편 b')
slope_play = widgets.Play(value=6, min=-20, max=20, step=1, interval=700)
slope_link = widgets.jslink((slope_play, 'value'), (slope_slider, 'value'))
fixed_output = widgets.interactive_output(compare_fixed, {'slope': slope_slider, 'intercept': fixed_b})
display(widgets.VBox([widgets.HBox([slope_play, slope_slider]), fixed_b]), fixed_output)

## 10. $\min_{w,b}J(w,b)$ · 기울기와 절편을 함께 고르기
이번에는 기울기뿐 아니라 직선의 위아래 위치도 고릅니다. 오른쪽 가로축은 $w$, 세로축은 $b$입니다. **지도 위 점 하나가 직선 하나**이고, 밝을수록 비용이 작습니다.

$b=0$만 고르는 것은 지도 맨 아래 가로줄에서만 찾는 일입니다. 둘 다 고르면 지도 전체에서 찾을 수 있습니다. 같은 비용 공식을 쓰지만 선택할 수 있는 직선의 범위가 다릅니다.

In [ ]:
# @title 두 매개변수의 비용 지도 준비
w_grid = np.linspace(-20, 20, 121)
b_grid = np.linspace(0, 90, 121)
W, B = np.meshgrid(w_grid, b_grid)
surface = np.mean((W[..., None] * x + B[..., None] - y) ** 2, axis=-1) / 2

def compare_joint(slope, intercept):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    draw_predictions(axes[0], slope, intercept)
    levels = np.geomspace(max(surface.min() * .9, .01), surface.max(), 25)
    contour = axes[1].contourf(W, B, surface, levels=levels, cmap='Blues')
    axes[1].axhline(0, color='black', linestyle='--', linewidth=2)
    axes[1].scatter([slope], [intercept], color=ORANGE, edgecolor='black', s=85, zorder=3)
    axes[1].set(xlabel='기울기 w', ylabel='절편 b', title=f'현재 비용 J={cost(slope, intercept):.2f} · 밝을수록 작음')
    fig.colorbar(contour, ax=axes[1], label='비용 J')
    plt.show()

### 실험 B · 두 손잡이를 따로 움직이기
먼저 기울기만 움직이고, 다음에는 절편만 움직여 보세요. 마지막으로 둘을 함께 조절해 비용을 줄여 보세요. 정답을 맞히기보다 **어느 조작이 직선을 어떻게 바꾸는지** 말하는 것이 목표입니다.

In [ ]:
joint_w = widgets.FloatSlider(value=6, min=-20, max=20, step=.2, description='기울기 w', continuous_update=True)
joint_b = widgets.FloatSlider(value=0, min=0, max=90, step=.5, description='절편 b', continuous_update=True)
joint_output = widgets.interactive_output(compare_joint, {'slope': joint_w, 'intercept': joint_b})
display(widgets.HBox([joint_w, joint_b]), joint_output)

## 11. 내가 찾은 직선과 최소 비용 비교하기
아래 셀은 비교용으로 최소 비용을 계산합니다. 계산 방법 자체는 아직 외울 필요 없습니다. 기울기만 고를 때도 허용되는 범위에서 가장 좋은 값을 사용해 공정하게 비교합니다.

In [ ]:
best_w_fixed = (x @ y) / (x @ x)
best_w, best_b = np.linalg.lstsq(np.column_stack([x, np.ones(m)]), y, rcond=None)[0]
display(pd.DataFrame({'선택 조건': ['b=0 고정, w만 선택', 'w와 b 모두 선택'], 'w': [best_w_fixed, best_w], 'b': [0, best_b], '최소 비용': [cost(best_w_fixed, 0), cost(best_w, best_b)]}).round(3))

$w,b$를 모두 고르는 경우에는 $b=0$인 직선도 선택할 수 있습니다. 따라서 가능한 최소 비용은 $b=0$으로 고정했을 때보다 커질 수 없습니다. 데이터에 따라 두 최소 비용이 같을 수도 있습니다.

$J(w)$라는 표기만 보고 언제나 $b=0$이라고 생각하면 안 됩니다. 이 실습에서는 **어떤 $b$를 고정했는지** 함께 표시했습니다.

### 확인 질문
1. 예측이 실제보다 작을 때 오차가 음수인 이유를 설명해 보세요.
2. 제곱 오차의 합계를 8로 나누고 2로 나누는 이유는 각각 무엇인가요?
3. 왼쪽 직선 하나가 오른쪽 비용 곡선의 점 하나가 되는 이유는 무엇인가요?
4. 기울기만 고르기와 기울기·절편을 함께 고르기의 차이는 무엇인가요?

비용이 줄었다는 것은 지금 사용한 8대에 더 잘 맞았다는 뜻입니다. 새로운 자동차에도 잘 맞는지는 이후 수업에서 별도로 확인합니다.